In [3]:
# 1. Install libraries
!pip install -q transformers sacrebleu pandas torch sentencepiece underthesea

# 2. Imports
import pandas as pd
import torch
import time
import sacrebleu
from transformers import MarianTokenizer, MarianMTModel
from underthesea import word_tokenize
from google.colab import drive


In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# 3. Load dataset
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Alignment/JSON2_4653_5815.csv' # đảm bảo file đã upload
df = pd.read_csv(DATA_PATH)

# 4. Select first 50 sentences as dev set
dev_df = df.head(50).copy()

sources = dev_df["src_lang"].astype(str).tolist()   # Chinese
references = dev_df["tgt_lang"].astype(str).tolist()  # Vietnamese

print(f"Loaded {len(sources)} dev sentence pairs")

Loaded 50 dev sentence pairs


In [6]:
# 5. Load MarianMT pretrained model
MODEL_NAME = "Helsinki-NLP/opus-mt-zh-vi"
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("Model loaded on:", device)

# 6. Translation + timing
translations = []
start_time = time.time()

for sent in sources:
    inputs = tokenizer(sent, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_length=128)
    translated = tokenizer.decode(output[0], skip_special_tokens=True)
    translations.append(translated)

end_time = time.time()

total_time = end_time - start_time
avg_time = total_time / len(sources)

# 7. Vietnamese tokenization (for BLEU)
def vi_tokenize(sent):
    return word_tokenize(sent, format="text")

hyp_tok = [vi_tokenize(h) for h in translations]
ref_tok = [vi_tokenize(r) for r in references]

# 8. BLEU (PRIMARY METRIC – sacrebleu)
bleu = sacrebleu.corpus_bleu(
    hyp_tok,
    [ref_tok],
    tokenize="none"
)

# 9. ChrF (OPTIONAL)
chrf = sacrebleu.corpus_chrf(
    translations,
    [references]
)

# 10. TER (OPTIONAL)
ter = sacrebleu.corpus_ter(
    translations,
    [references]
)

# 11. Save outputs for qualitative analysis
result_df = pd.DataFrame({
    "Chinese (src_lang)": sources,
    "Reference_VI (tgt_lang)": references,
    "MarianMT_Output": translations
})

result_df.to_csv("/content/drive/MyDrive/Colab Notebooks/Alignment/dev_translation_results.csv", index=False)

# 12. Print final report
print(f"Model           : {MODEL_NAME}")
print(f"Total time      : {total_time:.2f} seconds")
print(f"Avg / sentence  : {avg_time:.3f} seconds")
print(f"BLEU (primary)  : {bleu.score:.2f}")
print(f"ChrF            : {chrf.score:.2f}")
print(f"TER             : {ter.score:.2f}")
print("File save as: dev_translation_results.csv")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/750k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/766k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Model loaded on: cuda
Model           : Helsinki-NLP/opus-mt-zh-vi
Total time      : 30.39 seconds
Avg / sentence  : 0.608 seconds
BLEU (primary)  : 14.33
ChrF            : 36.01
TER             : 70.46
File save as: dev_translation_results.csv
